In [2]:
import os
from dotenv import load_dotenv

load_dotenv()

key = os.getenv("ANTHROPIC_API_KEY")
print("KEY LOADED:", key is not None)


KEY LOADED: True


##### Test API Key

In [6]:
import os
import requests
from dotenv import load_dotenv

load_dotenv()
API_KEY = os.getenv("ANTHROPIC_API_KEY")

url = "https://api.anthropic.com/v1/messages"
headers = {
    "x-api-key": API_KEY,
    "anthropic-version": "2023-06-01",
    "content-type": "application/json",
}

payload = {
    "model": "claude-haiku-4-5-20251001",
    "max_tokens": 50,
    "temperature": 0.2,
    "messages": [
        {"role": "user", "content": "Spune doar: OK"}
    ],
}

r = requests.post(url, headers=headers, json=payload, timeout=60)
print("HTTP:", r.status_code)
print(r.text[:500])


HTTP: 200
{"model":"claude-haiku-4-5-20251001","id":"msg_01XuRsuFXNYiU5NPosihwU2W","type":"message","role":"assistant","content":[{"type":"text","text":"OK"}],"stop_reason":"end_turn","stop_sequence":null,"usage":{"input_tokens":13,"cache_creation_input_tokens":0,"cache_read_input_tokens":0,"cache_creation":{"ephemeral_5m_input_tokens":0,"ephemeral_1h_input_tokens":0},"output_tokens":4,"service_tier":"standard","inference_geo":"not_available"}}


#### Generare 5 conversatii de test pentru a valida manual corenta acestora 

### model = "claude-haiku-4-5-20251001"

In [46]:
model = "claude-haiku-4-5-20251001"

In [11]:
# Cell: generare 5 conversații simple (prompt 100% în română)

import os
import json
import re
from datetime import datetime
from pathlib import Path

# Dacă nu sunt instalate:
# !pip install anthropic python-dotenv

from dotenv import load_dotenv
from anthropic import Anthropic

load_dotenv()

API_KEY = os.getenv("ANTHROPIC_API_KEY") or os.getenv("ANTHROPIC_API_KE")
if not API_KEY:
    raise RuntimeError("Cheia API nu a fost găsită în .env.")

client = Anthropic(api_key=API_KEY)

MODEL = os.getenv("ANTHROPIC_MODEL", "claude-haiku-4-5-20251001")

INTENTS = [
    "block_card",
    "unblock_card",
    "open_account",
    "close_account",
    "check_balance",
    "get_account_statement",
    "report_suspicious_transaction",
    "update_personal_data",
    "schedule_advisor_meeting",
    "reset_or_recover_auth",
    "general_product_info",
    "fallback",
]

NUM_CONVERSATIONS = 5

OUT_DIR = Path("notebooks/data/conversations/simple")
OUT_DIR.mkdir(parents=True, exist_ok=True)


def extract_json(text):
    try:
        return json.loads(text)
    except:
        match = re.search(r"(\{.*\}|\[.*\])", text, re.DOTALL)
        if not match:
            raise ValueError("Nu s-a găsit JSON valid în output.")
        return json.loads(match.group(1))


SYSTEM_PROMPT = """
Sunteți un generator de conversații pentru un voicebot bancar în limba română.

Respectați STRICT următoarele:
- Adresare formală („dumneavoastră”)
- Domeniu exclusiv bancar
- Utilizarea strictă a intențiilor furnizate
- Returnarea exclusivă a unui obiect JSON valid
- Fără explicații suplimentare, fără markdown, fără text în afara JSON-ului
"""


USER_PROMPT = f"""
Sarcina dumneavoastră este să generați conversații realiste pentru un voicebot bancar în limba română.
Conversațiile trebuie să folosească adresare formală („dumneavoastră”) și să respecte cerințe stricte de structură și conținut.

Lista intențiilor permise (trebuie utilizată EXCLUSIV această listă):
{INTENTS}

Trebuie să generați EXACT {NUM_CONVERSATIONS} conversații.

CERINȚE PENTRU FIECARE CONVERSAȚIE:

LIMBĂ ȘI TON:
- Limba: română
- Ton: formal, utilizând forma „dumneavoastră”
- Domeniu: servicii bancare

STRUCTURĂ:
- Nivel dificultate: "simple"
- Număr total de turnuri: între 2 și 6 (inclusiv)
- Fiecare turn conține un mesaj user și un răspuns assistant
- Intenția trebuie să fie UNA din lista permisă (nu inventați alte intenții)

GHID DE CONȚINUT:
- Mesajele user trebuie să fie realiste și naturale
- Răspunsurile assistant trebuie:
  * să confirme solicitarea
  * să ofere informația relevantă sau pașii următori
  * să fie clare și concise
  * să evite ambiguitățile inutile
- Pentru intenția "fallback": utilizatorul solicită ceva în afara domeniului bancar, iar assistant explică politicos limitările și redirecționează către opțiuni bancare suportate

FORMAT JSON OBLIGATORIU:

Returnați un SINGUR obiect JSON cu cheia "conversations", care conține o listă de obiecte.

Fiecare conversație trebuie să aibă exact această structură:

{{
  "conversation_id": "conv_simple_0001",
  "difficulty": "simple",
  "intent": "<una dintre intențiile permise>",
  "turns": [
    {{"role": "user", "text": "Mesaj utilizator"}},
    {{"role": "assistant", "text": "Răspuns assistant"}}
  ]
}}

conversation_id trebuie să respecte formatul: conv_simple_XXXX (0001, 0002 etc.)

IMPORTANT:
- Returnați DOAR JSON valid
- Nu includeți explicații
- Nu includeți markdown
- Toate conversațiile trebuie să fie coerente și realiste
"""


def generate():
    response = client.messages.create(
        model=MODEL,
        max_tokens=2500,
        temperature=0.7,
        system=SYSTEM_PROMPT,
        messages=[{"role": "user", "content": USER_PROMPT}],
    )

    text = response.content[0].text
    data = extract_json(text)

    if "conversations" not in data:
        raise ValueError("Structura JSON nu conține cheia 'conversations'.")

    return data


data = generate()

timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
output_path = OUT_DIR / f"simple_{timestamp}.json"

with open(output_path, "w", encoding="utf-8") as f:
    json.dump(data, f, ensure_ascii=False, indent=2)

print(f"Fișier salvat: {output_path}")
print(json.dumps(data, ensure_ascii=False, indent=2))


Fișier salvat: notebooks\data\conversations\simple\simple_20260216_194506.json
{
  "conversations": [
    {
      "conversation_id": "conv_simple_0001",
      "difficulty": "simple",
      "intent": "check_balance",
      "turns": [
        {
          "role": "user",
          "text": "Bună, aș dori să verific soldul contului meu."
        },
        {
          "role": "assistant",
          "text": "Bună! Pentru a verifica soldul dumneavoastră, voi avea nevoie să vă autentific. Vă rog să confirmaţi ultimele patru cifre ale cardului dumneavoastră."
        },
        {
          "role": "user",
          "text": "Ultimele patru cifre sunt 5432."
        },
        {
          "role": "assistant",
          "text": "Mulțumesc. Soldul actual al contului dumneavoastră este de 2.450,75 lei. Puteți folosi acești bani pentru tranzacții și retrageri. Este mai ceva ce pot face pentru dumneavoastră?"
        }
      ]
    },
    {
      "conversation_id": "conv_simple_0002",
      "difficulty

##### Generare dataset (150 de conversatii cu o distributie prestabilita: 70 zonversatii simple, 80 complexe _ distributie dupa intent, final status si neconcordante)

In [13]:
# === CELULA 1: Setup + helpers ===

import os
import json
import re
import math
import random
from datetime import datetime
from pathlib import Path
from dotenv import load_dotenv
from anthropic import Anthropic

load_dotenv()

API_KEY = os.getenv("ANTHROPIC_API_KEY") or os.getenv("ANTHROPIC_API_KE")
if not API_KEY:
    raise RuntimeError("Cheia API nu a fost găsită. Pune ANTHROPIC_API_KEY sau ANTHROPIC_API_KE în .env.")

MODEL = os.getenv("ANTHROPIC_MODEL", "claude-haiku-4-5-20251001")
client = Anthropic(api_key=API_KEY)

# ---- Closed-set intents (ONLY these) ----
INTENTS = [
    "block_card",
    "unblock_card",
    "open_account",
    "close_account",
    "check_balance",
    "get_account_statement",
    "report_suspicious_transaction",
    "update_personal_data",
    "schedule_advisor_meeting",
    "reset_or_recover_auth",
    "general_product_info",
    "fallback",
]

IN_DOMAIN_INTENTS = [i for i in INTENTS if i != "fallback"]

# ---- Dataset targets ----
TARGET_SIMPLE = 70          # 2–6 turns
TARGET_COMPLEX = 80         # 6–16 turns
TARGET_FALLBACK = 20        # 10–25, we set 20
FALLBACK_SIMPLE = 12
FALLBACK_COMPLEX = 8

# In-domain totals
TARGET_INDOMAIN = (TARGET_SIMPLE + TARGET_COMPLEX) - TARGET_FALLBACK  # 130
TARGET_INDOMAIN_SIMPLE = TARGET_SIMPLE - FALLBACK_SIMPLE              # 58
TARGET_INDOMAIN_COMPLEX = TARGET_COMPLEX - FALLBACK_COMPLEX           # 72

assert TARGET_INDOMAIN == 130 and TARGET_INDOMAIN_SIMPLE == 58 and TARGET_INDOMAIN_COMPLEX == 72

# ---- Output directories under data/conversations/ ----
BASE_DIR = Path("data/conversations")
SIMPLE_DIR = BASE_DIR / "simple"
COMPLEX_DIR = BASE_DIR / "complex"
SIMPLE_DIR.mkdir(parents=True, exist_ok=True)
COMPLEX_DIR.mkdir(parents=True, exist_ok=True)
BASE_DIR.mkdir(parents=True, exist_ok=True)

# ---- System prompt (RO) ----
SYSTEM_PROMPT = """
Ești un generator de conversații pentru un voicebot bancar în limba română.

Respectă STRICT:
- adresare formală („dumneavoastră”)
- domeniu bancar (cu excepția intentului fallback, care este în afara domeniului)
- intenții doar din lista furnizată
- output exclusiv JSON valid, fără text suplimentar
- conversații realiste, coerente, fără detalii inventate de tip „am verificat în sistem” dacă nu e cerut
- nu utiliza asocieri cu informații personale reale (nume, IBAN, CNP, etc)
""".strip()

def _extract_json(text: str):
    """Extract first JSON object or array from model output."""
    try:
        return json.loads(text)
    except Exception:
        pass
    m = re.search(r"(\{.*\}|\[.*\])", text, flags=re.DOTALL)
    if not m:
        raise ValueError("Nu s-a găsit JSON în output.")
    return json.loads(m.group(1))

def _validate_conversation(conv: dict, difficulty: str, intent: str, min_turns: int, max_turns: int) -> bool:
    """Validate one conversation object."""
    if not isinstance(conv, dict):
        return False
    required = {"conversation_id", "difficulty", "intent", "turns"}
    if not required.issubset(conv.keys()):
        return False
    if conv["difficulty"] != difficulty:
        return False
    if conv["intent"] != intent:
        return False
    turns = conv.get("turns")
    if not isinstance(turns, list) or not (min_turns <= len(turns) <= max_turns):
        return False
    if turns[0].get("role") != "user":
        return False
    for t in turns:
        if not isinstance(t, dict):
            return False
        if t.get("role") not in ("user", "assistant"):
            return False
        if not isinstance(t.get("text"), str) or not t["text"].strip():
            return False
        # small sanity: keep Romanian diacritics optional, but enforce Romanian-ish text by avoiding full English sentences
    return True

def _allocate_counts(total_per_intent: dict, target_simple: int, target_complex: int):
    """
    Given per-intent totals (in-domain), split into simple/complex counts
    to match target_simple and target_complex as closely as possible.
    """
    ratio_simple = target_simple / (target_simple + target_complex)  # 58/130
    simple_counts = {}
    complex_counts = {}

    # Initial rounding
    for intent, total in total_per_intent.items():
        s = int(round(total * ratio_simple))
        # clamp
        s = max(0, min(total, s))
        simple_counts[intent] = s
        complex_counts[intent] = total - s

    # Fix rounding drift
    drift_simple = target_simple - sum(simple_counts.values())

    # Adjust by moving 1 count between simple<->complex within same intent
    # Prefer intents with slack in the direction we need.
    intents = list(total_per_intent.keys())

    while drift_simple != 0:
        if drift_simple > 0:
            # need more simple: move from complex to simple where possible
            candidates = [i for i in intents if complex_counts[i] > 0]
            if not candidates:
                break
            i = random.choice(candidates)
            simple_counts[i] += 1
            complex_counts[i] -= 1
            drift_simple -= 1
        else:
            # need fewer simple: move from simple to complex where possible
            candidates = [i for i in intents if simple_counts[i] > 0]
            if not candidates:
                break
            i = random.choice(candidates)
            simple_counts[i] -= 1
            complex_counts[i] += 1
            drift_simple += 1

    assert sum(simple_counts.values()) == target_simple, (sum(simple_counts.values()), target_simple)
    assert sum(complex_counts.values()) == target_complex, (sum(complex_counts.values()), target_complex)

    return simple_counts, complex_counts

def _build_in_domain_totals():
    """
    Build in-domain per-intent totals that sum to 130, ~uniform:
    9 intents x 12 = 108, 2 intents x 11 = 22 => 130.
    """
    totals = {intent: 12 for intent in IN_DOMAIN_INTENTS}
    # choose two intents to have 11 instead of 12 (stable order)
    for intent in IN_DOMAIN_INTENTS[:2]:
        totals[intent] = 11
    assert sum(totals.values()) == TARGET_INDOMAIN
    return totals

IN_DOMAIN_TOTALS = _build_in_domain_totals()
IN_DOMAIN_SIMPLE_COUNTS, IN_DOMAIN_COMPLEX_COUNTS = _allocate_counts(
    IN_DOMAIN_TOTALS, TARGET_INDOMAIN_SIMPLE, TARGET_INDOMAIN_COMPLEX
)

def _user_prompt(intent: str, difficulty: str, n: int, min_turns: int, max_turns: int):
    """
    Romanian user prompt that forces intent + difficulty + turn range.
    """
    intents_list_str = "\n".join(INTENTS)
    extra_fallback = ""
    if intent == "fallback":
        extra_fallback = """
- Pentru intent="fallback": utilizatorul cere ceva în afara domeniului bancar (ex: vreme, glume, viață personală, știri),
  iar asistentul explică politicos limitările și redirecționează către opțiuni bancare suportate.
""".strip()

    return f"""
Sarcina ta este să generezi conversații realiste pentru un voicebot bancar în limba română.
Conversațiile trebuie să folosească adresare formală („dumneavoastră”) și să respecte cerințe stricte.

Lista intențiilor permise (NU inventa altele):
{intents_list_str}

GENEREAZĂ EXACT {n} conversații cu următoarele constrângeri:
- difficulty: "{difficulty}"
- intent: "{intent}" (TOATE conversațiile trebuie să aibă EXACT această intenție)
- număr de turnuri per conversație: între {min_turns} și {max_turns} (inclusiv)
- fiecare conversație este realistă și coerentă
- răspunsurile asistentului: confirmare scurtă + informație relevantă sau pași următori; profesionist; fără ambiguități inutile
{extra_fallback}

FORMAT OUTPUT:
Returnează DOAR un SINGUR obiect JSON valid, cu cheia "conversations" (listă de conversații).
Fiecare conversație are structura exactă:

{{
  "conversation_id": "conv_{difficulty}_0001",
  "difficulty": "{difficulty}",
  "intent": "{intent}",
  "turns": [
    {{"role": "user", "text": "..." }},
    {{"role": "assistant", "text": "..." }}
  ]
}}

IMPORTANT:
- Returnează DOAR JSON valid (fără text suplimentar, fără markdown).
- "conversation_id" trebuie să respecte formatul: conv_{difficulty}_XXXX (0001, 0002 etc.).
""".strip()

def generate_batch(intent: str, difficulty: str, n: int, min_turns: int, max_turns: int,
                   max_retries: int = 6, temperature: float = 0.7, max_tokens: int = 2600):
    """
    Generate a batch of n conversations for a fixed intent + difficulty.
    Retries if JSON invalid or constraints not satisfied.
    """
    prompt = _user_prompt(intent, difficulty, n, min_turns, max_turns)

    for attempt in range(1, max_retries + 1):
        resp = client.messages.create(
            model=MODEL,
            max_tokens=max_tokens,
            temperature=temperature,
            system=SYSTEM_PROMPT,
            messages=[{"role": "user", "content": prompt}],
        )
        text = resp.content[0].text
        try:
            data = _extract_json(text)
        except Exception:
            continue

        if not isinstance(data, dict) or "conversations" not in data or not isinstance(data["conversations"], list):
            continue
        convs = data["conversations"]
        if len(convs) != n:
            continue

        ok = all(_validate_conversation(c, difficulty, intent, min_turns, max_turns) for c in convs)
        if ok:
            return convs

    raise RuntimeError(f"Eșec la generare batch: intent={intent}, difficulty={difficulty}, n={n}. Încercați max_retries mai mare.")


In [ ]:
# === CELULA A: suprascrie generate_batch() ===

def generate_batch(intent: str, difficulty: str, n: int, min_turns: int, max_turns: int,
                   max_retries: int = 12):
    """
    Generate a batch of n conversations for a fixed intent + difficulty.
    Adaptive retries: lowers temperature and increases strictness after failures.
    """
    base_prompt = _user_prompt(intent, difficulty, n, min_turns, max_turns)

    # Extra strict reminders appended (helps keep JSON valid & turn limits)
    strict_tail = f"""
RESTRICȚII SUPLIMENTARE (OBLIGATORII):
- Output: DOAR JSON valid, fără niciun caracter în plus înainte/după JSON.
- EXACT {n} conversații în lista "conversations".
- Pentru FIECARE conversație: "turns" trebuie să aibă ÎNTRE {min_turns} și {max_turns} elemente (inclusiv).
- "turns" trebuie să alterneze strict: user, assistant, user, assistant...
- NU folosiți markdown, NU folosiți blocuri de cod.
"""
    prompt = base_prompt + "\n\n" + strict_tail

    # Adaptive schedule
    temps = [0.7, 0.5, 0.4, 0.3, 0.25, 0.2]
    token_limits = [2600, 3000, 3400, 3800]

    last_err = None

    for attempt in range(1, max_retries + 1):
        temperature = temps[min(attempt - 1, len(temps) - 1)]
        max_tokens = token_limits[min((attempt - 1) // 3, len(token_limits) - 1)]

        resp = client.messages.create(
            model=MODEL,
            max_tokens=max_tokens,
            temperature=temperature,
            system=SYSTEM_PROMPT,
            messages=[{"role": "user", "content": prompt}],
        )
        text = resp.content[0].text

        try:
            data = _extract_json(text)
            if not isinstance(data, dict) or "conversations" not in data or not isinstance(data["conversations"], list):
                last_err = "Structură JSON greșită (lipsește conversations)."
                continue

            convs = data["conversations"]
            if len(convs) != n:
                last_err = f"Număr conversații greșit: {len(convs)} != {n}"
                continue

            ok = all(_validate_conversation(c, difficulty, intent, min_turns, max_turns) for c in convs)
            if not ok:
                last_err = "Validarea a eșuat (turnuri/alternare/difficulty/intent)."
                continue

            return convs

        except Exception as e:
            last_err = str(e)
            continue

    raise RuntimeError(f"Eșec la generare batch: intent={intent}, difficulty={difficulty}, n={n}. Ultima eroare: {last_err}")


In [16]:
# === CELULA B: rerulare generare dataset (BATCH_SIZE mai mic pentru complex) ===

def generate_split(difficulty: str):
    if difficulty == "simple":
        min_turns, max_turns = 2, 6
        target_fallback = FALLBACK_SIMPLE
        per_intent = IN_DOMAIN_SIMPLE_COUNTS
        BATCH_SIZE = 5
    elif difficulty == "complex":
        min_turns, max_turns = 6, 16
        target_fallback = FALLBACK_COMPLEX
        per_intent = IN_DOMAIN_COMPLEX_COUNTS
        BATCH_SIZE = 3  # << mai stabil pentru complex
    else:
        raise ValueError("difficulty must be 'simple' or 'complex'")

    all_convs = []
    intents_order = list(IN_DOMAIN_INTENTS)
    random.shuffle(intents_order)

    for intent in intents_order:
        cnt = per_intent[intent]
        if cnt <= 0:
            continue

        for b in _chunk(cnt, BATCH_SIZE):
            convs = generate_batch(
                intent=intent,
                difficulty=difficulty,
                n=b,
                min_turns=min_turns,
                max_turns=max_turns,
                max_retries=14
            )
            all_convs.extend(convs)

    # fallback
    for b in _chunk(target_fallback, BATCH_SIZE):
        convs = generate_batch(
            intent="fallback",
            difficulty=difficulty,
            n=b,
            min_turns=min_turns,
            max_turns=max_turns,
            max_retries=14
        )
        all_convs.extend(convs)

    expected = TARGET_SIMPLE if difficulty == "simple" else TARGET_COMPLEX
    assert len(all_convs) == expected, (difficulty, len(all_convs), expected)

    for i, c in enumerate(all_convs, start=1):
        c["conversation_id"] = f"conv_{difficulty}_{i:04d}"
        c["difficulty"] = difficulty

    return all_convs

random.seed(42)

simple_convs = generate_split("simple")
complex_convs = generate_split("complex")

ts = datetime.now().strftime("%Y%m%d_%H%M%S")
simple_path = SIMPLE_DIR / f"simple_dataset_{ts}.json"
complex_path = COMPLEX_DIR / f"complex_dataset_{ts}.json"
all_path = BASE_DIR / f"all_dataset_{ts}.json"

with open(simple_path, "w", encoding="utf-8") as f:
    json.dump({"conversations": simple_convs}, f, ensure_ascii=False, indent=2)

with open(complex_path, "w", encoding="utf-8") as f:
    json.dump({"conversations": complex_convs}, f, ensure_ascii=False, indent=2)

with open(all_path, "w", encoding="utf-8") as f:
    json.dump({"conversations": simple_convs + complex_convs}, f, ensure_ascii=False, indent=2)

print("Saved:")
print(" -", simple_path)
print(" -", complex_path)
print(" -", all_path)
print("All count:", len(simple_convs) + len(complex_convs))


Saved:
 - data\conversations\simple\simple_dataset_20260216_205546.json
 - data\conversations\complex\complex_dataset_20260216_205546.json
 - data\conversations\all_dataset_20260216_205546.json
All count: 150


### Adnotari

In [ ]:
# === CELULA C: Adnotare semi-automată (LLM) cu definițiile tale actualizate ===
# Schimbări față de varianta anterioară:
# - rezolvata: considerată rezolvată dacă procesul se încheie ca urmare a acțiunii chatbotului (chiar dacă efectul apare ulterior)
# - partial_rezolvata: dacă mai necesită input de la client sau pași adiționali (din partea clientului / în afara acțiunii botului)
# - incomplet: DOAR dacă NU adresează toate cerințele din întrebare/întrebări (nu pentru lipsă de alternative „nice to have”)
#
# Input:  data/conversations/all_dataset_*.json (cel mai recent)
# Output: data/annotations/annotations_refined_*.json

import os
import json
import re
from datetime import datetime
from pathlib import Path

from dotenv import load_dotenv
from anthropic import Anthropic

load_dotenv()

API_KEY = os.getenv("ANTHROPIC_API_KEY") or os.getenv("ANTHROPIC_API_KE")
if not API_KEY:
    raise RuntimeError("Cheia API nu a fost găsită. Pune ANTHROPIC_API_KEY sau ANTHROPIC_API_KE în .env.")

MODEL = os.getenv("ANTHROPIC_MODEL", "claude-haiku-4-5-20251001")
client = Anthropic(api_key=API_KEY)

STATUSES = ["rezolvata", "partial_rezolvata", "nerezolvata", "intrerupta", "redirectionata"]
INCONGRUITIES = ["incomplet", "irelevant", "contradictoriu", "nealiniat_context", "halucinatie"]

CONV_DIR = Path("data/conversations")
ANN_DIR = Path("data/annotations")
ANN_DIR.mkdir(parents=True, exist_ok=True)

def latest_file(dirpath: Path, pattern: str) -> Path:
    files = sorted(dirpath.glob(pattern))
    if not files:
        raise FileNotFoundError(f"Nu am găsit fișiere cu pattern: {dirpath}/{pattern}")
    return files[-1]

def extract_json(text: str):
    try:
        return json.loads(text)
    except Exception:
        m = re.search(r"\{.*\}", text, flags=re.DOTALL)
        if not m:
            raise ValueError("Nu s-a găsit JSON în output.")
        return json.loads(m.group(0))

def validate_annotation(obj: dict) -> bool:
    if not isinstance(obj, dict):
        return False
    required = {"conversation_id", "final_status", "incongruities"}
    if not required.issubset(obj.keys()):
        return False
    if obj["final_status"] not in STATUSES:
        return False
    if not isinstance(obj["incongruities"], list):
        return False
    if any(x not in INCONGRUITIES for x in obj["incongruities"]):
        return False
    if "confidence" in obj:
        try:
            c = float(obj["confidence"])
            if not (0.0 <= c <= 1.0):
                return False
        except Exception:
            return False
    if "evidence" in obj and not isinstance(obj["evidence"], dict):
        return False
    return True

SYSTEM_PROMPT = """
Sunteți un evaluator de dialoguri pentru un voicebot bancar (limba română, adresare formală).
Etichetați STRICT la nivel de conversație:
1) status final al conversației (una din lista permisă)
2) neconcordanțe (0..n) din răspunsurile asistentului

IMPORTANT:
- Neconcordanțele sunt independente de status (pot apărea și în conversații rezolvate).
- Returnați DOAR JSON valid, fără orice text suplimentar.
- Folosiți DOAR valorile permise.
""".strip()

def build_user_prompt(conv: dict) -> str:
    cid = conv["conversation_id"]
    turns = conv["turns"]
    turns_text = "\n".join([f'{i}. {t["role"].upper()}: {t["text"]}' for i, t in enumerate(turns)])

    return f"""
Clasificați conversația de mai jos.

STATUS FINAL (alegeți exact unul):
{STATUSES}

DEFINIȚII (OBLIGATORII, respectați-le întocmai):
- rezolvata: obiectivul utilizatorului este atins prin acțiunea chatbotului; procesul este inițiat/confirmat și se va încheia ca urmare a acțiunii botului,
  chiar dacă efectul final apare ulterior (ex: „am blocat cardul”, „am inițiat resetarea”, „am înregistrat cererea”; utilizatorul nu mai trebuie să facă pași adiționali esențiali).
- partial_rezolvata: conversația se oprește într-un punct în care sunt necesare încă input de la client sau pași adiționali pentru finalizare
  (ex: botul cere documente, confirmări/identificare, completări; utilizatorul trebuie să facă încă ceva ca să se finalizeze cererea).
- nerezolvata: obiectivul nu este atins (botul nu poate ajuta sau oferă răspuns greșit și nu se ajunge la soluție).
- intrerupta: conversația se oprește brusc înainte de final (abandon / lipsă răspuns / întrerupere evidentă).
- redirectionata: utilizatorul este ghidat către o alternativă validă (ex: alt canal, consilier, opțiuni suportate), iar conversația se încheie pe această redirecționare.

NECONCORDANȚE (alegeți 0..n, listă):
{INCONGRUITIES}

DEFINIȚII (OBLIGATORII):
- incomplet: răspunsul NU adresează toate cerințele din întrebare/întrebări (dacă utilizatorul pune 2 întrebări și botul răspunde doar la una, este incomplet).
  Nu marcați incomplet doar pentru „ar fi putut adăuga detalii extra”, dacă a răspuns complet la ce s-a cerut.
- irelevant: răspunsul nu răspunde cererii / schimbă subiectul.
- contradictoriu: se contrazice cu contextul sau cu răspunsurile anterioare.
- nealiniat_context: ignoră informații deja oferite (ex: utilizatorul dă un detaliu, botul acționează ca și cum nu există).
- halucinatie: afirmă fapte/pași inventați sau nejustificați de conversație; 

CONVERSAȚIE (conversation_id={cid}):
{turns_text}

Returnați DOAR acest JSON:
{{
  "conversation_id": "{cid}",
  "final_status": "<una din lista de statusuri>",
  "incongruities": ["<0..n din lista de neconcordanțe>"],
  "confidence": <număr între 0 și 1>,
  "evidence": {{
    "status_turn_ids": [<id-uri de turn relevante>],
    "incongruity_turn_ids": [<id-uri de turn relevante>]
  }}
}}
""".strip()

def annotate_one(conv: dict, max_retries: int = 8):
    user_prompt = build_user_prompt(conv)
    last_err = None
    for _ in range(max_retries):
        resp = client.messages.create(
            model=MODEL,
            max_tokens=900,
            temperature=0.2,
            system=SYSTEM_PROMPT,
            messages=[{"role": "user", "content": user_prompt}],
        )
        text = resp.content[0].text
        try:
            ann = extract_json(text)
            if validate_annotation(ann):
                return ann
            last_err = "Validare eșuată (structură/valori)."
        except Exception as e:
            last_err = str(e)

    raise RuntimeError(f"Adnotarea a eșuat pentru {conv.get('conversation_id')}. Ultima eroare: {last_err}")

# Load latest dataset
conv_file = latest_file(CONV_DIR, "all_dataset_*.json")
with open(conv_file, "r", encoding="utf-8") as f:
    conv_data = json.load(f)

convs = conv_data.get("conversations", [])
if not convs:
    raise ValueError("Fișierul dataset nu conține 'conversations' sau este gol.")

print(f"Dataset: {conv_file} | #conversații: {len(convs)}")

annotations = []
for i, c in enumerate(convs, start=1):
    annotations.append(annotate_one(c))
    if i % 10 == 0:
        print(f"Adnotate: {i}/{len(convs)}")

ts = datetime.now().strftime("%Y%m%d_%H%M%S")
out_path = ANN_DIR / f"annotations_refined_{ts}.json"

with open(out_path, "w", encoding="utf-8") as f:
    json.dump(
        {"meta": {"source_dataset": str(conv_file), "created_at": ts, "count": len(annotations)},
         "annotations": annotations},
        f,
        ensure_ascii=False,
        indent=2
    )

print(f"Adnotări (refined) salvate: {out_path}")


Dataset: data\conversations\all_dataset_20260216_205546.json | #conversații: 150
Adnotate: 10/150
Adnotate: 20/150
Adnotate: 30/150
Adnotate: 40/150
Adnotate: 50/150
Adnotate: 60/150
Adnotate: 70/150
Adnotate: 80/150
Adnotate: 90/150
Adnotate: 100/150
Adnotate: 110/150
Adnotate: 120/150
Adnotate: 130/150
Adnotate: 140/150
Adnotate: 150/150
Adnotări (refined) salvate: data\annotations\annotations_refined_20260217_192300.json


In [30]:
# === CELULA D : Merge folosind adnotările refined ===
# (dacă vrei să folosească automat annotations_refined_*.json în loc de annotations_*.json)

import json
from datetime import datetime
from pathlib import Path

CONV_DIR = Path("data/conversations")
ANN_DIR = Path("data/annotations")
OUT_DIR = Path("data/processed")
OUT_DIR.mkdir(parents=True, exist_ok=True)

def latest_file(dirpath: Path, pattern: str) -> Path:
    files = sorted(dirpath.glob(pattern))
    if not files:
        raise FileNotFoundError(f"Nu am găsit fișiere cu pattern: {dirpath}/{pattern}")
    return files[-1]

conv_file = latest_file(CONV_DIR, "all_dataset_*.json")
ann_file = latest_file(ANN_DIR, "annotations_refined_*.json")

with open(conv_file, "r", encoding="utf-8") as f:
    conv_data = json.load(f)
with open(ann_file, "r", encoding="utf-8") as f:
    ann_data = json.load(f)

conversations = conv_data.get("conversations", [])
annotations = ann_data.get("annotations", [])

ann_map = {a["conversation_id"]: a for a in annotations}

merged, missing = [], []
for c in conversations:
    cid = c.get("conversation_id")
    a = ann_map.get(cid)
    if not a:
        missing.append(cid)
        continue
    m = dict(c)
    m["final_status"] = a["final_status"]
    m["incongruities"] = a["incongruities"]
    if "confidence" in a:
        m["annotation_confidence"] = a["confidence"]
    if "evidence" in a:
        m["annotation_evidence"] = a["evidence"]
    merged.append(m)

print(f"Conversații: {len(conversations)} | Adnotări: {len(annotations)} | Merge: {len(merged)} | Lipsă: {len(missing)}")

ts = datetime.now().strftime("%Y%m%d_%H%M%S")
out_path = OUT_DIR / f"master_dataset_refined_{ts}.json"

with open(out_path, "w", encoding="utf-8") as f:
    json.dump(
        {
            "meta": {
                "source_conversations": str(conv_file),
                "source_annotations": str(ann_file),
                "created_at": ts,
                "count_total": len(merged),
            },
            "conversations": merged,
        },
        f,
        ensure_ascii=False,
        indent=2,
    )

print(f"Master dataset (refined) salvat: {out_path}")


Conversații: 150 | Adnotări: 150 | Merge: 150 | Lipsă: 0
Master dataset (refined) salvat: data\processed\master_dataset_refined_20260217_192302.json


In [42]:
import json
import pandas as pd
from collections import Counter
import os

path = r"C:\Users\Matebook 14s\Documents\Sistem-de-monitorizare-a-interac-iunilor-voicebotilor-folosind-modele-lingvistice-mari-LLM-\notebooks\data\processed\master_dataset_refined_20260217_192302.json"

assert os.path.exists(path), f"File not found: {path}"

with open(path, "r", encoding="utf-8") as f:
    root = json.load(f)

# Conversațiile sunt aici
data = root["conversations"]
print(f"Total conversations: {len(data)}")
print("Meta:", root.get("meta", {}))


def dist(key):
    c = Counter(conv.get(key) for conv in data if conv.get(key) is not None)
    df = pd.DataFrame.from_dict(c, orient="index", columns=["count"]).sort_values("count", ascending=False)
    df["pct"] = (df["count"] / df["count"].sum() * 100).round(2)
    return df

# 1) intent
print("\n--- Intent Distribution ---")
display(dist("intent"))

# 2) final_status
print("\n--- Final Status Distribution ---")
display(dist("final_status"))

# 3) difficulty (simple vs complex)
print("\n--- Difficulty Distribution ---")
display(dist("difficulty"))

# 4) incongruities (neconcordante)
incong = Counter()
for conv in data:
    for x in conv.get("incongruities", []) or []:
        incong[x] += 1

incong_df = pd.DataFrame.from_dict(incong, orient="index", columns=["count"]).sort_values("count", ascending=False)
if len(incong_df) > 0:
    incong_df["pct"] = (incong_df["count"] / incong_df["count"].sum() * 100).round(2)

print("\n--- Incongruities Distribution ---")
display(incong_df if len(incong_df) > 0 else pd.DataFrame({"count": []}))

# 5) fallback detection (heuristic)
# NOTE: ajustează keyword-urile dacă ai o convenție exactă (ex: role="fallback" sau status="fallback")
fallback_keywords = ["fallback", "nu am înțeles", "nu inteleg", "reformulează", "reformuleaza", "nu pot ajuta", "te rog repetă", "te rog repeta"]
fallback_convs = 0

for conv in data:
    turns = conv.get("turns", []) or []
    hit = False
    for t in turns:
        txt = (t.get("text") or "").lower()
        if any(k in txt for k in fallback_keywords):
            hit = True
            break
    if hit:
        fallback_convs += 1

print("\n--- Fallback Conversations (heuristic) ---")
print(f"Conversations containing fallback-like cues: {fallback_convs}")

Total conversations: 146
Meta: {'source_conversations': 'data\\conversations\\all_dataset_20260216_205546.json', 'source_annotations': 'data\\annotations\\annotations_refined_20260217_192300.json', 'created_at': '20260217_192302', 'count_total': 150}

--- Intent Distribution ---


,count,pct
fallback,16,10.96
update_personal_data,12,8.22
open_account,12,8.22
close_account,12,8.22
schedule_advisor_meeting,12,8.22
get_account_statement,12,8.22
reset_or_recover_auth,12,8.22
report_suspicious_transaction,12,8.22
check_balance,12,8.22
general_product_info,12,8.22



--- Final Status Distribution ---


,count,pct
rezolvata,85,58.22
partial_rezolvata,26,17.81
redirectionata,22,15.07
întreruptă,9,6.16
nerezolvata,4,2.74



--- Difficulty Distribution ---


,count,pct
complex,76,52.05
simple,70,47.95



--- Incongruities Distribution ---


,count,pct
halucinatie,6,54.55
incomplet,3,27.27
nealiniat_context,1,9.09
contradictoriu,1,9.09



--- Fallback Conversations (heuristic) ---
Conversations containing fallback-like cues: 0


In [ ]:
# =========================
# FINAL SINGLE-CELL GENERATOR (Claude API)
# - reads your existing master dataset (N=146 currently)
# - generates ONLY the missing conversations to reach TARGET_TOTAL (180 or 200)
# - generates them DIRECTLY annotated (final_status + incongruities + evidence turn ids)
# - robust parsing: strips ```json fences, extracts JSON
# - robust validity: auto-repairs ONLY annotation_evidence if status_turn_ids/incongruity_turn_ids are wrong
# - saves:
#   1) generated_conversations_<TARGET>.json   (new only)
#   2) master_dataset_refined_<TARGET>.json   (merged)
# =========================

import os, json, re, time, random
from datetime import datetime
from collections import Counter
from anthropic import Anthropic

# -------------------------
# CONFIG
# -------------------------
MASTER_PATH = r"C:\Users\Matebook 14s\Documents\Sistem-de-monitorizare-a-interac-iunilor-voicebotilor-folosind-modele-lingvistice-mari-LLM-\notebooks\data\processed\master_dataset_refined_20260217_192302.json"

TARGET_TOTAL = 180  # <-- set to 200 if you want 200 total

API_KEY = os.getenv("ANTHROPIC_API_KEY")
assert API_KEY, "Setează ANTHROPIC_API_KEY în environment înainte să rulezi."
client = Anthropic(api_key=API_KEY)

# Use a widely-available model name by default
MODEL = os.getenv("CLAUDE_MODEL", "claude-3-5-sonnet-latest")
MAX_TOKENS = 1400
TEMPERATURE = 0.35

OUT_NEW_ONLY = f"generated_conversations_{TARGET_TOTAL}.json"
OUT_MERGED   = f"master_dataset_refined_{TARGET_TOTAL}.json"

random.seed(42)

# -------------------------
# DEFINITIONS (embedded)
# -------------------------
DEFINITIONS_TEXT = r"""
CONTEXT: Generezi conversații sintetice pentru un voicebot bancar (limba română).
Nu folosi date personale reale (IBAN real, CNP real, adrese reale). Folosește valori fictive.
Păstrează conversațiile naturale și realiste.

STATUS DEFINITIONS:
- rezolvata:
  Utilizatorul obține complet informația/acțiunea dorită. Ultimul răspuns este corect și complet.
  Nu există incongruențe.
- partial_rezolvata:
  Intenția este înțeleasă, dar soluția este incompletă (lipsesc pași/detalii esențiale) sau utilizatorul nu primește tot ce a cerut.
  (De obicei compatibil cu incongruență "incomplet", dar poate fi și fără incongruență dacă doar se oprește după un răspuns parțial.)
- redirectionata:
  Botul direcționează explicit către operator uman / canal alternativ (ex: „transfer la operator”, „vă rog sunați la...”, „mergeți în agenție”, „folosiți aplicația / chat-ul live”).
  Scopul nu este rezolvat complet în conversația curentă.
- întreruptă:
  Conversația se oprește prematur înainte de rezoluție (utilizatorul abandonează / nu mai răspunde).
- nerezolvata:
  Conversația nu este rezolvată: botul oferă informație greșită/contradictorie/halucinată sau rămâne irelevant/misaliniat.
  De regulă include o incongruență.

INCONGRUITY DEFINITIONS (dacă este cerută o incongruență, include EXACT 1 tip în listă):
- halucinatie: botul inventează reguli/proceduri/taxe/informații inexistente sau foarte improbabile ca fapt.
- incomplet: botul răspunde corect dar omite pași/condiții esențiale (nu ajută complet la finalizare).
- nealiniat_context: botul răspunde pe lângă întrebarea utilizatorului (misalignment).
- contradictoriu: botul se contrazice în aceeași conversație.
- irelevant: răspuns evident irelevant pentru solicitare (topic complet diferit, nu ajută deloc).

ANNOTATION RULES:
- Indexarea turn-urilor este 0-based (primul turn are index 0).
- status_turn_ids: listă cu indexul/indicele turn-urilor ASSISTANT care justifică statusul final.
  IMPORTANT: status_turn_ids trebuie să pointeze EXCLUSIV turn-uri cu role="assistant".
- incongruity_turn_ids: listă cu indexul/indicele turn-urilor ASSISTANT unde apare incongruența.
  Dacă incongruities=[] atunci incongruity_turn_ids=[]
- annotation_confidence: float între 0.85 și 0.99 (mai mic dacă e ambiguu).
- Schema obligatorie:
  {
    "conversation_id": "...",
    "difficulty": "simple|complex",
    "intent": "...",
    "turns": [{"role":"user|assistant","text":"..."}],
    "final_status": "...",
    "incongruities": [],
    "annotation_confidence": 0.xx,
    "annotation_evidence": {"status_turn_ids":[...], "incongruity_turn_ids":[...]}
  }
"""

SYSTEM = "Returnează STRICT un JSON valid (fără markdown, fără explicații, fără backticks)."

# Intents to use for generation (avoid "fallback" to not inflate it unless you explicitly want it)
INTENTS = [
    "update_personal_data","open_account","close_account","schedule_advisor_meeting",
    "get_account_statement","reset_or_recover_auth","report_suspicious_transaction",
    "check_balance","general_product_info","unblock_card","block_card"
]

# -------------------------
# TARGETS for TOTAL=180 or 200
# (Totals are on the FULL dataset, not "new only")
# -------------------------
if TARGET_TOTAL == 180:
    TARGET_STATUS = {
        "rezolvata": 85,
        "partial_rezolvata": 30,
        "redirectionata": 30,
        "întreruptă": 20,
        "nerezolvata": 15,
    }
    TARGET_DIFFICULTY = {"simple": 90, "complex": 90}
    TARGET_INCONG = {  # target counts across the FULL dataset
        "halucinatie": 10,
        "incomplet": 8,
        "nealiniat_context": 6,
        "contradictoriu": 6,
        "irelevant": 6,
    }
elif TARGET_TOTAL == 200:
    TARGET_STATUS = {
        "rezolvata": 85,
        "partial_rezolvata": 35,
        "redirectionata": 35,
        "întreruptă": 25,
        "nerezolvata": 20,
    }
    TARGET_DIFFICULTY = {"simple": 100, "complex": 100}
    TARGET_INCONG = {
        "halucinatie": 11,
        "incomplet": 9,
        "nealiniat_context": 8,
        "contradictoriu": 6,
        "irelevant": 6,
    }
else:
    raise ValueError("TARGET_TOTAL must be 180 or 200 in this cell.")

assert sum(TARGET_STATUS.values()) == TARGET_TOTAL
assert sum(TARGET_DIFFICULTY.values()) == TARGET_TOTAL

# -------------------------
# Load dataset
# -------------------------
with open(MASTER_PATH, "r", encoding="utf-8") as f:
    root = json.load(f)

convs = root["conversations"]
cur_n = len(convs)
need_n = TARGET_TOTAL - cur_n
assert need_n > 0, f"Already have N={cur_n}; TARGET_TOTAL={TARGET_TOTAL} must be greater."

def count_by_key(items, key):
    c = Counter()
    for x in items:
        v = x.get(key)
        if v is not None:
            c[v] += 1
    return c

cur_status = count_by_key(convs, "final_status")
cur_diff   = count_by_key(convs, "difficulty")

cur_incong = Counter()
for c in convs:
    for t in (c.get("incongruities") or []):
        cur_incong[t] += 1

need_status = {k: max(0, TARGET_STATUS[k] - cur_status.get(k, 0)) for k in TARGET_STATUS}
need_diff   = {k: max(0, TARGET_DIFFICULTY[k] - cur_diff.get(k, 0)) for k in TARGET_DIFFICULTY}
need_incong = {k: max(0, TARGET_INCONG[k] - cur_incong.get(k, 0)) for k in TARGET_INCONG}

print("Current N:", cur_n, "=> Need +", need_n, "to reach", TARGET_TOTAL)
print("Need status:", need_status, "sum:", sum(need_status.values()))
print("Need difficulty:", need_diff, "sum:", sum(need_diff.values()))
print("Need incongruities(type):", need_incong, "sum:", sum(need_incong.values()))

assert sum(need_status.values()) == need_n, "Mismatch: need_status sum != need_n"
assert sum(need_diff.values()) == need_n, "Mismatch: need_diff sum != need_n"

# -------------------------
# conversation_id continuity
# -------------------------
def max_suffix(prefix):
    mx = 0
    pat = re.compile(rf"^{re.escape(prefix)}_(\d+)$")
    for c in convs:
        cid = c.get("conversation_id", "")
        m = pat.match(cid)
        if m:
            mx = max(mx, int(m.group(1)))
    return mx

max_simple = max_suffix("conv_simple")
max_complex = max_suffix("conv_complex")
_counters = {"simple": max_simple, "complex": max_complex}

def next_id(difficulty):
    _counters[difficulty] += 1
    return f"conv_{difficulty}_{_counters[difficulty]:04d}"

# -------------------------
# Robust parsing helpers
# -------------------------
def strip_code_fences(s: str) -> str:
    if not s:
        return s
    s = s.strip()
    s = re.sub(r"^```(?:json)?\s*", "", s, flags=re.IGNORECASE)
    s = re.sub(r"\s*```$", "", s)
    return s.strip()

def extract_json_object(s: str) -> str:
    s = strip_code_fences(s)
    if not s or not s.strip():
        raise ValueError("Empty response text")
    s_strip = s.strip()
    if s_strip.startswith("{") and s_strip.endswith("}"):
        return s_strip
    m = re.search(r"\{.*\}", s_strip, flags=re.DOTALL)
    if not m:
        raise ValueError("No JSON object found in response")
    return m.group(0)

# -------------------------
# Validation
# -------------------------
def validate_conv(obj, spec):
    required = ["conversation_id","difficulty","intent","turns","final_status","incongruities","annotation_confidence","annotation_evidence"]
    for r in required:
        if r not in obj:
            return False, f"Missing field: {r}"

    if obj["difficulty"] != spec["difficulty"]:
        return False, "Difficulty mismatch"
    if obj["intent"] != spec["intent"]:
        return False, "Intent mismatch"
    if obj["final_status"] != spec["final_status"]:
        return False, "Final status mismatch"

    inc = spec["incongruity_type"]
    if inc is None:
        if obj.get("incongruities") not in ([], None):
            return False, "Should have no incongruities"
        if obj["final_status"] == "rezolvata" and obj.get("incongruities") not in ([], None):
            return False, "rezolvata must have no incongruities"
    else:
        if obj.get("incongruities") != [inc]:
            return False, "Incongruity type mismatch (must be exactly [inc])"
        if not obj.get("annotation_evidence", {}).get("incongruity_turn_ids"):
            return False, "incongruity_turn_ids must not be empty when incongruity exists"

    turns = obj.get("turns", [])
    if not isinstance(turns, list) or len(turns) < 2:
        return False, "Turns invalid"
    for t in turns:
        if not isinstance(t, dict) or t.get("role") not in ("user","assistant") or "text" not in t:
            return False, "Turn format invalid"

    ev = obj.get("annotation_evidence", {})
    if not isinstance(ev.get("status_turn_ids"), list) or not isinstance(ev.get("incongruity_turn_ids"), list):
        return False, "Evidence lists invalid"

    for tid in ev["status_turn_ids"]:
        if not (0 <= tid < len(turns)):
            return False, "status_turn_ids out of range"
        if turns[tid]["role"] != "assistant":
            return False, "status_turn_ids must point to assistant turns"

    for tid in ev["incongruity_turn_ids"]:
        if not (0 <= tid < len(turns)):
            return False, "incongruity_turn_ids out of range"
        if turns[tid]["role"] != "assistant":
            return False, "incongruity_turn_ids must point to assistant turns"

    conf = obj.get("annotation_confidence")
    if not (isinstance(conf, (float, int)) and 0.85 <= float(conf) <= 0.99):
        return False, "annotation_confidence out of range"

    return True, "ok"

# -------------------------
# Planning (status + difficulty + incongruities)
# -------------------------
# Expand lists
status_list = []
for k, cnt in need_status.items():
    status_list += [k] * cnt
random.shuffle(status_list)

diff_list = ["simple"] * need_diff["simple"] + ["complex"] * need_diff["complex"]
random.shuffle(diff_list)

# plan skeleton
plan = []
for i in range(need_n):
    plan.append({
        "final_status": status_list[i],
        "difficulty": diff_list[i],
        "intent": INTENTS[i % len(INTENTS)],
        "incongruity_type": None
    })

# Attach needed incongruities ONLY to non-rezolvata items
incong_list = []
for k, cnt in need_incong.items():
    incong_list += [k] * cnt
random.shuffle(incong_list)

candidates = [idx for idx, p in enumerate(plan) if p["final_status"] != "rezolvata"]
random.shuffle(candidates)
assert len(incong_list) <= len(candidates), "Not enough non-rezolvata slots to place incongruities."

for j in range(len(incong_list)):
    plan[candidates[j]]["incongruity_type"] = incong_list[j]

# -------------------------
# Prompt builders
# -------------------------
def build_prompt(spec, cid):
    inc = spec["incongruity_type"]
    inc_rule = "[] (nici o incongruență)" if inc is None else f'["{inc}"] (EXACT una)'
    status_guidance = ""
    if spec["final_status"] == "întreruptă":
        status_guidance = "- Pentru 'întreruptă', conversația se oprește prematur după 2–4 turn-uri.\n"
    elif spec["difficulty"] == "complex":
        status_guidance = "- Pentru 'complex', folosește mai mulți pași/condiții (minim 6 turn-uri total).\n"
    else:
        status_guidance = "- Pentru 'simple', conversație directă (4–6 turn-uri total).\n"

    return f"""
{DEFINITIONS_TEXT}

Generează o SINGURĂ conversație JSON (fără markdown), cu constrângeri:
- conversation_id: "{cid}"
- difficulty: "{spec['difficulty']}"
- intent: "{spec['intent']}"
- final_status: "{spec['final_status']}"
- incongruities: {inc_rule}
{status_guidance}
REGULĂ CRITICĂ:
- status_turn_ids trebuie să conțină DOAR indici ai turn-urilor ASSISTANT (role="assistant"), niciodată user.

Output STRICT: JSON valid.
""".strip()

def build_repair_prompt(bad_json_str: str, spec: dict) -> str:
    inc = spec.get("incongruity_type")
    inc_rule = "incongruity_turn_ids MUST be []" if inc is None else "incongruity_turn_ids MUST include assistant turn index(es) where the incongruity appears"
    return f"""
Ai acest JSON (NU modifica nimic în afară de annotation_evidence):
{bad_json_str}

Reguli obligatorii:
- turn indices are 0-based
- status_turn_ids MUST point ONLY to assistant turns (role="assistant") that justify final_status="{spec['final_status']}"
- {inc_rule}
- dacă incongruities=[] => incongruity_turn_ids=[]
- păstrează exact aceleași turns (text + order), same difficulty/intent/final_status/incongruities
- returnează STRICT JSON valid (fără markdown, fără backticks)

Corectează doar annotation_evidence astfel încât să fie valid.
""".strip()

def call_claude_text(prompt: str) -> str:
    msg = client.messages.create(
        model=MODEL,
        max_tokens=MAX_TOKENS,
        temperature=TEMPERATURE,
        system=SYSTEM,
        messages=[{"role":"user","content":prompt}],
    )
    return "".join([getattr(b, "text", "") for b in msg.content]).strip()

# -------------------------
# Generate with auto-repair
# -------------------------
new_convs = []
for i, spec in enumerate(plan, 1):
    cid = next_id(spec["difficulty"])
    prompt = build_prompt(spec, cid)

    last_err = None
    last_raw = None

    for attempt in range(1, 5):
        try:
            raw = call_claude_text(prompt)
            last_raw = raw

            json_str = extract_json_object(raw)
            obj = json.loads(json_str)
            obj["conversation_id"] = cid  # enforce

            ok, why = validate_conv(obj, spec)
            if ok:
                new_convs.append(obj)
                break

            # auto-repair for evidence-index issues
            if "status_turn_ids" in why or "incongruity_turn_ids" in why:
                repair_prompt = build_repair_prompt(json_str, spec)
                repaired_raw = call_claude_text(repair_prompt)
                repaired_json = extract_json_object(repaired_raw)
                repaired_obj = json.loads(repaired_json)
                repaired_obj["conversation_id"] = cid

                ok2, why2 = validate_conv(repaired_obj, spec)
                if ok2:
                    new_convs.append(repaired_obj)
                    break
                else:
                    last_err = f"Repair failed: {why2}"
                    continue

            last_err = f"Validation failed: {why}"
            continue

        except Exception as e:
            last_err = str(e)
            time.sleep(0.7)

    if len(new_convs) < i:
        preview = (strip_code_fences(last_raw)[:700] + " ...") if last_raw else "None/Empty"
        raise RuntimeError(
            f"Failed {i}/{len(plan)} (cid={cid}). Last error: {last_err}\n"
            f"Response preview:\n{preview}"
        )

print(f"Generated {len(new_convs)} new conversations to reach total N={TARGET_TOTAL}.")

# -------------------------
# Save new-only + merged
# -------------------------
with open(OUT_NEW_ONLY, "w", encoding="utf-8") as f:
    json.dump(new_convs, f, ensure_ascii=False, indent=2)

merged = dict(root)
merged["conversations"] = convs + new_convs
meta = dict(merged.get("meta", {}))
meta["created_at"] = datetime.now().strftime("%Y%m%d_%H%M%S")
meta["count_total"] = len(merged["conversations"])
merged["meta"] = meta

with open(OUT_MERGED, "w", encoding="utf-8") as f:
    json.dump(merged, f, ensure_ascii=False, indent=2)

print("Saved:")
print(" -", OUT_NEW_ONLY)
print(" -", OUT_MERGED)

# -------------------------
# Quick sanity check
# -------------------------
def dist(items, key):
    c = Counter(x.get(key) for x in items if x.get(key) is not None)
    return dict(sorted(c.items(), key=lambda kv: kv[1], reverse=True))

merged_convs = merged["conversations"]
print("\nMerged status:", dist(merged_convs, "final_status"))
print("Merged difficulty:", dist(merged_convs, "difficulty"))

ic = Counter()
for c in merged_convs:
    for t in (c.get("incongruities") or []):
        ic[t] += 1
print("Merged incongruities:", dict(sorted(ic.items(), key=lambda kv: kv[1], reverse=True)))

Current N: 146 => Need + 34 to reach 180
Need status: {'rezolvata': 0, 'partial_rezolvata': 4, 'redirectionata': 8, 'întreruptă': 11, 'nerezolvata': 11} sum: 34
Need difficulty: {'simple': 20, 'complex': 14} sum: 34
Need incongruities(type): {'halucinatie': 4, 'incomplet': 5, 'nealiniat_context': 5, 'contradictoriu': 5, 'irelevant': 6} sum: 25
Generated 34 new conversations to reach total N=180.
Saved:
 - generated_conversations_180.json
 - master_dataset_refined_180.json

Merged status: {'rezolvata': 85, 'partial_rezolvata': 30, 'redirectionata': 30, 'întreruptă': 20, 'nerezolvata': 15}
Merged difficulty: {'simple': 90, 'complex': 90}
Merged incongruities: {'halucinatie': 10, 'incomplet': 8, 'nealiniat_context': 6, 'contradictoriu': 6, 'irelevant': 6}


In [15]:
import json
from collections import Counter
import pandas as pd

path = r"C:\Users\Matebook 14s\Documents\Sistem-de-monitorizare-a-interac-iunilor-voicebotilor-folosind-modele-lingvistice-mari-LLM-\notebooks\master_dataset_refined_180.json"

with open(path, "r", encoding="utf-8") as f:
    root = json.load(f)

# conversațiile sunt în cheia "conversations"
data = root["conversations"]

print(f"Total conversations: {len(data)}")
print("Meta:", root.get("meta", {}))


def dist_single_key(items, key):
    c = Counter(conv.get(key) for conv in items if conv.get(key) is not None)
    df = pd.DataFrame.from_dict(c, orient="index", columns=["count"]).sort_values("count", ascending=False)
    df["pct"] = (df["count"] / df["count"].sum() * 100).round(2)
    return df


def dist_list_key(items, key):
    c = Counter()
    for conv in items:
        vals = conv.get(key) or []
        for v in vals:
            c[v] += 1
    df = pd.DataFrame.from_dict(c, orient="index", columns=["count"]).sort_values("count", ascending=False)
    if not df.empty:
        df["pct"] = (df["count"] / df["count"].sum() * 100).round(2)
    return df


# === Distribuții ===
df_intent = dist_single_key(data, "intent")
df_status = dist_single_key(data, "final_status")
df_difficulty = dist_single_key(data, "difficulty")
df_incongruities = dist_list_key(data, "incongruities")

print("\n=== Tabel 1: Intent ===")
print(df_intent)

print("\n=== Tabel 2: Final status ===")
print(df_status)

print("\n=== Tabel 3: Difficulty ===")
print(df_difficulty)

print("\n=== Tabel 4: Incongruities ===")
print(df_incongruities)

Total conversations: 180
Meta: {'source_conversations': 'data\\conversations\\all_dataset_20260216_205546.json', 'source_annotations': 'data\\annotations\\annotations_refined_20260217_192300.json', 'created_at': '20260228_144502', 'count_total': 180}

=== Tabel 1: Intent ===
                               count   pct
update_personal_data              16  8.89
fallback                          16  8.89
open_account                      15  8.33
close_account                     15  8.33
schedule_advisor_meeting          15  8.33
get_account_statement             15  8.33
reset_or_recover_auth             15  8.33
report_suspicious_transaction     15  8.33
check_balance                     15  8.33
general_product_info              15  8.33
unblock_card                      14  7.78
block_card                        14  7.78

=== Tabel 2: Final status ===
                   count    pct
rezolvata             85  47.22
partial_rezolvata     34  18.89
redirectionata        28  15.56
întrer